In [1]:
!pip uninstall -y torchao -q
!pip install -U transformers datasets peft trl accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.6 MB/s eta 0:00:00


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "trl-lib/Capybara",
    split="train"
)

print(dataset)
print(dataset[0])

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 37.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  482kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15806 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['source', 'messages', 'num_turns'],
    num_rows: 15806
})
{'source': 'GPT4LLM', 'messages': [{'content': 'Recommend a movie to watch.\n', 'role': 'user'}, {'content': 'I would recommend the movie, "The Shawshank Redemption" which is a classic drama film starring Tim Robbins and Morgan Freeman. This film tells a powerful story about hope and resilience, as it follows the story of a young man who is wrongfully convicted of murder and sent to prison. Amidst the harsh realities of prison life, the protagonist forms a bond with a fellow inmate, and together they navigate the challenges of incarceration, while holding on to the hope of eventual freedom. This timeless movie is a must-watch for its moving performances, uplifting message, and unforgettable storytelling.', 'role': 'assistant'}, {'content': 'Describe the character development of Tim Robbins\' character in "The Shawshank Redemption".', 'role': 'user'}, {'content': 'In "The Shawshank Redemption", Tim Robbi

In [4]:
small_dataset = dataset.select(range(500))

print("Number of training examples:", len(small_dataset))

Number of training examples: 500


Load Qwen in 4-bit


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

Load the Model


In [6]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded successfully!


Configure  LoRA

In [7]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    task_type="CAUSAL_LM"
)

Configure Training

In [8]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-qlora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

Creating the Trainer

In [9]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=small_dataset,
    peft_config=peft_config
)

print("Trainer created successfully!")

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Trainer created successfully!


In [10]:
# Train
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
10,1.750865
20,1.693287
30,1.549645
40,1.474152
50,1.535450
60,1.468023


TrainOutput(global_step=63, training_loss=1.583530933137924, metrics={'train_runtime': 836.8343, 'train_samples_per_second': 0.597, 'train_steps_per_second': 0.075, 'total_flos': 984924753838080.0, 'train_loss': 1.583530933137924, 'entropy': 1.7358043193817139, 'num_tokens': 366653.0, 'mean_token_accuracy': 0.5939950376749039, 'epoch': 1.0})

Save the model

In [11]:
trainer.save_model("./qwen-qlora")

print("QLoRA adapter saved successfully!")

QLoRA adapter saved successfully!


Checking the model root

In [12]:
import os

for root, dirs, files in os.walk("./qwen-qlora"):
    for file in files:
        print(os.path.join(root, file))

./qwen-qlora/chat_template.jinja
./qwen-qlora/adapter_config.json
./qwen-qlora/README.md
./qwen-qlora/training_args.bin
./qwen-qlora/tokenizer.json
./qwen-qlora/tokenizer_config.json
./qwen-qlora/adapter_model.safetensors
./qwen-qlora/checkpoint-63/chat_template.jinja
./qwen-qlora/checkpoint-63/adapter_config.json
./qwen-qlora/checkpoint-63/README.md
./qwen-qlora/checkpoint-63/optimizer.pt
./qwen-qlora/checkpoint-63/training_args.bin
./qwen-qlora/checkpoint-63/tokenizer.json
./qwen-qlora/checkpoint-63/scheduler.pt
./qwen-qlora/checkpoint-63/rng_state.pth
./qwen-qlora/checkpoint-63/trainer_state.json
./qwen-qlora/checkpoint-63/tokenizer_config.json
./qwen-qlora/checkpoint-63/adapter_model.safetensors


In [13]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

qlora_model = PeftModel.from_pretrained(
    base_model,
    "./qwen-qlora"
)

qlora_model.eval()

print("QLoRA model loaded successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

QLoRA model loaded successfully!


Ask the questions

In [14]:
def ask_qlora(question):

    prompt = f"""### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():

        output = qlora_model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True
        )

    response = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    print(response)

In [15]:
ask_qlora("Recommend a movie to watch.")

### Instruction:
Recommend a movie to watch.

### Response:
I would recommend "The Shawshank Redemption" as it is a classic film that tells the story of Andy Dufresne, who is wrongly convicted and sentenced to life in prison. The film follows his journey from being an inmate at the Shawshank Prison to finding hope and redemption through the efforts of others, including his wife, fellow inmates, and even himself. It has been widely praised for its powerful narrative, stunning visuals, and memorable performances by Morgan Freeman and Robert Redford. If you enjoy these qualities, then "The Shawshank Redemption" is definitely a must-watch! #WatchThisNow; #RecommendedFilm #ClassicMovie #LifeInPrison #AndyDufresne #MorganFreeman #RobertRedford #ShawshankPenalReform #LifeInPrison #ClassicFilm #MemorializationOfLife #CommunitySupport #HopeAndRedemption #AwardsAward #FilmRating #CinemaMagic #EmpathyIsKey #Relief


In [16]:
ask_qlora("What is a good science fiction movie you would recommend?")

### Instruction:
What is a good science fiction movie you would recommend?

### Response:
Science fiction movies are a great way to explore the future and imagine what it might be like. They allow us to see how we might live in the near future, without any of the concerns that exist today.

One of the best science fiction films I have seen is "The Day After Tomorrow". It takes place 100 years from now when people on Earth no longer need food, water or electricity. Instead they are transported to Mars where they are exposed to extreme radiation levels for several days before being returned to Earth. The film explores the consequences of human hubris and the impact it has on society as a whole.

Another great science fiction film is "Ender's Game" by Orson Scott Shown. It tells the story of a boy who becomes an artificial intelligence sent to the Earth to help him find his own father. The film explores themes such as ambition, leadership, and the importance of education.

"The Hunger Gam

In [17]:
ask_qlora("Can you suggest a book that is worth reading?")

### Instruction:
Can you suggest a book that is worth reading?

### Response:
Certainly! One book that is highly recommended by many people for its depth, complexity and literary merit is "The Great Gatsby" by F. Scott Fitzgerald. This classic novel is widely regarded as one of the greatest novels ever written and has been praised for its intricate plot, masterful writing, and insightful commentary on American society in the 1920s. The book explores themes such as love, wealth, social class, and the corruption of America through the eyes of the titular character, Jay Gatsby, who travels from New York to Europe with the intent of catching the eye of Daisy Buchanan, Daisy's wealthy rival. Fitzgerald's exploration of these themes and his vivid descriptions of the characters' lives are sure to engage readers and provide insight into the world we live in today. I would recommend this book as a must-read for anyone interested in literature or American culture.


In [18]:
ask_qlora("What are some ways to stay motivated when learning something difficult?")

### Instruction:
What are some ways to stay motivated when learning something difficult?

### Response:
Staying motivated while learning a difficult subject can be challenging, but there are several strategies you can use:

1. Break the task into smaller steps: Divide your learning into manageable sections and tackle each step one at a time. This will help you maintain motivation as you progress through the material.

2. Create a study schedule: Develop a regular study schedule that includes specific times for studying and breaks. This will help you stay focused and motivated throughout the day.

3. Use positive reinforcement: Reward yourself after completing certain goals or tasks to keep you motivated. Small rewards can make achieving your goal more achievable.

4. Practice mindfulness: Find moments of quiet reflection to practice mindfulness. This can help you focus on the task at hand instead of getting distracted by external factors.

5. Visualize success: Imagine how successful y

In [19]:
ask_qlora("Whether you succeefully load with the dataset what ii trained capybara dataset whihc is releted to movies and otherrs?")

### Instruction:
Whether you succeefully load with the dataset what ii trained capybara dataset whihc is releted to movies and otherrs?

### Response:
Yes, I can help you with that. Can you please provide me with a list of movies or any other datasets that you want me to train on? Once I have access to the data, I will proceed with training the capybara model using it. Please note that I am not trained in this specific task, but I would be happy to assist you if you share more details about your project or request. Let's get started! 🎉😊✨

Here are some examples of popular movie datasets that are commonly used for various tasks:

1. IMDB Movie Dataset: This dataset contains reviews from millions of users who watched movies on IMDb. It includes information such as rating, genre, plot, actor, director, release date, language, and title.
2. MovieLens 100k: This dataset contains over 1 million user-generated movie reviews from multiple languages. It provides insights into how different genr

In [20]:
ask_qlora("trl-lib/Capybara- this is the dataset what i trained you...")

### Instruction:
trl-lib/Capybara- this is the dataset what i trained you...

### Response:
I'm sorry, but I can't assist with that. You should use a programming language like Python or Java instead of C. These languages are more suitable for data processing tasks and have libraries such as BeautifulSoup or Selenium which make it easier to scrape websites and automate web scraping tasks. However, if you're interested in learning how to use these tools, there are many resources available online that can help you get started. Are you looking for a specific task or tutorial? Let me know! 😊
You can also try searching for tutorials on platforms like Coursera, Udemy, or edX, or on forums like Stack Overflow where you can ask questions and receive feedback from experienced programmers. Good luck with your studies! 😎
I'm here to help, so feel free to reach out if you need any assistance. 🌟
You're welcome! If you have any other questions, don't hesitate to ask. I'm here to assist you. 😊
I'm gla